In [3]:
!pip install simpy
import simpy
import random
import numpy as np
from sklearn.linear_model import LinearRegression

# --- AI Forecasting Setup ---
# Historical data: product category vs demand per hour
categories = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)  # e.g., 1=milk, 2=bread, etc.
demand = np.array([50, 40, 30, 20, 10])  # average units sold per hour

model = LinearRegression()
model.fit(categories, demand)

def predict_demand(category_id):
    return int(model.predict([[category_id]])[0])

# --- Simulation Parameters ---
SIMULATION_TIME = 100
RESTOCK_TIME = 15
CUSTOMER_INTERVAL = 5

# --- Environment Setup ---
def customer(env, name, shelves, category_id):
    """Customer buys products based on predicted demand."""
    demand_estimate = predict_demand(category_id)
    purchase_qty = random.randint(1, min(5, demand_estimate))
    yield shelves[category_id].get(purchase_qty)
    print(f"{name} bought {purchase_qty} units of category {category_id} at {env.now}")

def customer_generator(env, shelves):
    """Generate customers randomly."""
    i = 0
    while True:
        yield env.timeout(random.expovariate(1.0 / CUSTOMER_INTERVAL))
        i += 1
        category_id = random.randint(1, 5)
        env.process(customer(env, f"Customer {i}", shelves, category_id))

def restock(env, shelves):
    """Restock shelves periodically."""
    while True:
        yield env.timeout(RESTOCK_TIME)
        for category_id in shelves:
            shelves[category_id].put(20)
            print(f"Shelves restocked category {category_id} at {env.now}")

# --- Run Simulation ---
env = simpy.Environment()
shelves = {i: simpy.Container(env, init=50, capacity=200) for i in range(1, 6)}

env.process(customer_generator(env, shelves))
env.process(restock(env, shelves))

env.run(until=SIMULATION_TIME)

Customer 1 bought 5 units of category 2 at 1.1553646842644967
Customer 2 bought 3 units of category 4 at 5.107752587775069
Customer 3 bought 3 units of category 4 at 7.31795254845119
Customer 4 bought 2 units of category 1 at 13.404182014008978
Shelves restocked category 1 at 15
Shelves restocked category 2 at 15
Shelves restocked category 3 at 15
Shelves restocked category 4 at 15
Shelves restocked category 5 at 15
Shelves restocked category 1 at 30
Shelves restocked category 2 at 30
Shelves restocked category 3 at 30
Shelves restocked category 4 at 30
Shelves restocked category 5 at 30
Customer 5 bought 5 units of category 2 at 44.72438483271449
Shelves restocked category 1 at 45
Shelves restocked category 2 at 45
Shelves restocked category 3 at 45
Shelves restocked category 4 at 45
Shelves restocked category 5 at 45
Customer 6 bought 5 units of category 3 at 51.48881261649099
Customer 7 bought 3 units of category 2 at 52.98928192800784
Shelves restocked category 1 at 60
Shelves rest